# EDA Lending dataset, conforme a las indicaciones del profesor:

Construir un modelo de clasificación supervisada para predecir si un préstamo emitido por la plataforma Lending Club resultará en default (1) o será pagado completamente (0). El objetivo es comparar el desempeño de los modelos construidos con scikit-learn y PySpark, además de aplicar LIME para interpretar predicciones. Se espera que, bajo las condiciones establecidas en este proyecto, PySpark supere en velocidad a scikit-learn.

Así que empezamos con el EDA

In [ ]:
import os

data_aceptados = os.path.expanduser("~\\Desktop\\Stage\\Frontstage\\Creations\\ModelSolar\\notebooks\\PIDAA\\Dataset\\accepted_2007_to_2018q4.csv\\accepted_2007_to_2018Q4.csv")

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Imports basicos

In [ ]:
df = pd.read_csv(data_aceptados, low_memory=False)


9.10.4.1.1. Carga inicial y visión general

    Cargar el dataset CSV completo (sin muestreo) utilizando pandas y, opcionalmente, Spark para comparar tiempos de carga.

    Mostrar las primeras y últimas filas (head(), tail()) para verificar la correcta lectura.

    Usar info() y describe() para obtener un resumen de tipos de datos, valores nulos y estadísticas descriptivas básicas de las variables numéricas.

    Reportar la dimensión del dataset (número de filas y columnas).

9.10.4.1.2. Análisis unidimensional (por variable)

Para cada variable, según su tipo (numérica o categórica), se deben realizar las siguientes actividades:

Variables numéricas (ej. loan_amnt, int_rate, annual_inc, dti, fico_range_high, emp_length):

    Calcular y reportar: media, mediana, desviación estándar, mínimo, máximo, percentiles (25%, 50%, 75%) y rango intercuartil (IQR).

    Identificar posibles valores atípicos (outliers) mediante el método del IQR (valores fuera de [Q1 - 1.5IQR, Q3 + 1.5IQR]) y mediante inspección visual.

    Generar histogramas para visualizar la distribución de frecuencias.

    Generar diagramas de caja (boxplots) para identificar asimetría y presencia de outliers.

    Evaluar la normalidad de la distribución (mediante asimetría o prueba de Shapiro-Wilk opcional) y comentar si sería necesario aplicar transformaciones (log, Box-Cox) en etapas posteriores.

    Calcular la tasa de valores faltantes por variable y decidir un umbral (ej. >30%) para posible eliminación o imputación avanzada.

Variables categóricas (ej. purpose, home_ownership, addr_state, verification_status):

    Reportar la frecuencia absoluta y relativa de cada categoría.

    Identificar categorías con muy baja frecuencia (ej. menos del 1% de los datos) que podrían agruparse en una categoría “Otros”.

    Generar gráficos de barras horizontales o verticales para visualizar la distribución.

    Detectar si existen categorías con significado redundante o que puedan unificarse.

    Calcular la tasa de valores faltantes y considerar su tratamiento (moda, categoría “Desconocido”, o eliminación si es muy alta).

Variable objetivo (default):

    Calcular la distribución de clases (0 = Fully Paid, 1 = Charged Off) en valores absolutos y porcentajes.

    Visualizar con un gráfico de barras o pastel.

    Comentar si existe desbalance de clases y qué impacto podría tener en el modelado (por ejemplo, necesidad de usar estratificación, weighted metrics o técnicas de balanceo).

9.10.4.1.3. Análisis bidimensional (relaciones entre variables)

Relación de cada variable numérica con la variable objetivo default:

    Generar boxplots comparativos por clase (default = 0 vs 1) para cada variable numérica, con el fin de observar diferencias en la mediana, dispersión y presencia de outliers separados por clase.

    Generar violin plots (opcional pero recomendado) para visualizar la densidad de la distribución por clase.

    Calcular la diferencia de medias entre clases y realizar una prueba t de Student o prueba de Mann-Whitney (si no hay normalidad) para determinar si la diferencia es estadísticamente significativa (p-value < 0.05).

    Calcular la correlación punto-biserial entre cada variable numérica y la variable binaria default.

Relación de cada variable categórica con la variable objetivo default:

    Generar tablas de contingencia (crosstab) que muestren, para cada categoría, la proporción de default (tasa de impago).

    Visualizar mediante gráficos de barras apiladas o barras agrupadas la frecuencia de default por categoría.

    Realizar una prueba de chi-cuadrado para evaluar la independencia entre cada variable categórica y default. Reportar el estadístico y el p-value.

    Identificar categorías con alta tasa de default que podrían ser predictores relevantes.

Relaciones entre variables independientes (multicolinealidad):

    Calcular la matriz de correlaciones entre todas las variables numéricas usando el coeficiente de Pearson.

    Visualizar la matriz de correlación con un mapa de calor (heatmap) y comentar los pares de variables con correlación alta (|r| > 0.7) que podrían generar multicolinealidad.

    Para pares de variables categóricas, utilizar el coeficiente V de Cramer o la prueba de chi-cuadrado para evaluar asociación.

    Identificar posibles redundancias (ej. fico_range_high con int_rate) que justifiquen la exclusión de alguna variable en el modelado.

Visualizaciones avanzadas sugeridas:

    Pairplot (solo para una muestra representativa si el dataset es muy grande) de las principales variables numéricas coloreado por default.

    Gráficos de densidad (KDE) comparativos por clase para las variables más relevantes.

    Matriz de dispersión (scatter matrix) limitada a 4-5 variables clave para identificar patrones no lineales.

9.10.4.1.4. Valores faltantes (análisis detallado)

    Reportar, para cada columna, el número y porcentaje de valores nulos.

    Visualizar el patrón de missing values mediante un mapa de calor de nulos o una biblioteca como missingno.

    Analizar si los valores faltantes se concentran en ciertas filas o columnas, y si existe relación con la variable objetivo (prueba de missingness completamente al azar, MCAR).

    Proponer un plan de tratamiento diferenciado por variable: eliminación (si el porcentaje es muy alto, >70%), imputación con mediana/media/moda, o creación de una categoría “Missing”.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import missingno as msno

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 99

In [ ]:
df.shape

226-algo de filas y algunas columnas, poquitas, no muchas, definitivamente un dataset chiquitito,jaja

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df)) * 100
resumen_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos["n_nulos"] > 0].sort_values("pct_nulos", ascending=False)
resumen_nulos

#Valores faltantes por columna, tabla exigida en 9.10.4.1.4: columna, conteo, porcentaje.

In [ ]:
df.isnull().any(axis=1).sum() #Filas con al menos un NA

In [ ]:
df.isnull().all(axis=1).sum() #Filas completamente vacías en todas las columnas:

In [ ]:
umbral = 70
cols_eliminar = resumen_nulos[resumen_nulos["pct_nulos"] > umbral].index.tolist()
len(cols_eliminar)
cols_eliminar

Todas las columnas que salen en la tabla superior seran eliminadas ya que mas del 70% de sus valores son nulos

In [ ]:
df = df.drop(columns=cols_eliminar)
df.shape

hacemos lo mismo con el umbral del 50

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df)) * 100
resumen_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos["n_nulos"] > 0].sort_values("pct_nulos", ascending=False)
resumen_nulos

#Valores faltantes por columna, tabla exigida en 9.10.4.1.4: columna, conteo, porcentaje.

In [ ]:
umbral = 50
cols_eliminar = resumen_nulos[resumen_nulos["pct_nulos"] > umbral].index.tolist()
len(cols_eliminar)
cols_eliminar

Las eliminaremos, ya que cualquier metodo de imputacion seria basicamente adivinar.

In [ ]:
df = df.drop(columns=cols_eliminar)
df.shape

Revisemos el umbral de 30%

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df)) * 100
resumen_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos["n_nulos"] > 0].sort_values("pct_nulos", ascending=False)
resumen_nulos

#Valores faltantes por columna, tabla exigida en 9.10.4.1.4: columna, conteo, porcentaje.

In [ ]:
umbral = 30
cols_eliminar = resumen_nulos[resumen_nulos["pct_nulos"] > umbral].index.tolist()
len(cols_eliminar)
cols_eliminar

Parece que todas tienen que ver con fechas, revisemos como se ven esas columnas

In [ ]:
columnas_revision = ['il_util',
 'mths_since_rcnt_il',
 'all_util',
 'open_acc_6m',
 'inq_last_12m',
 'total_cu_tl',
 'open_act_il',
 'open_il_12m',
 'open_rv_12m',
 'total_bal_il',
 'open_rv_24m',
 'max_bal_bc',
 'inq_fi',
 'open_il_24m']

dfrevision = pd.read_csv(
    data_aceptados,
    usecols=columnas_revision,
    low_memory=False
)

dfrevision.head()

No tengo ni idea de que significan esas columnas, y como el unico diccionario estaba en una pagina que aparentemente cerraron¿?¿? es mejor no usar eso para el modelo https://data.world/jaypeedevlin/lending-club-loan-data-2007-11/workspace/file?filename=LCDataDictionary.csv.
hubiese sido util ughhh


Nota: (Sarcasticamente) De lo lindo


Revisemos el umbral del 15%

In [ ]:
umbral = 15
cols_eliminar = resumen_nulos[resumen_nulos["pct_nulos"] > umbral].index.tolist()
len(cols_eliminar)
cols_eliminar

exactamente las mismas... mejor se realiza eliminación


In [ ]:
df = df.drop(columns=cols_eliminar)
df.shape

Listo, quedamos con columnas que si nos podrian aportar informacion claramente o que serian manejables con inputación, ya que sus datos faltantes no superan el 15% de los datos

Empezamos con el analisis univariado y multivariado, no sin antes:

9.10.3. Target

    Variable binaria: loan_status

        0 = Fully Paid

        1 = Charged Off (default)

    Generar la variable default basada en la columna loan_status:

df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)


ese codigo esta raro..... huh bueno

In [ ]:
columna = df['loan_status']
columna.value_counts()

el codigo del profe hace q los datos sean irrelevantes en parte pq mete todo lo q no es "Charged Off" en la misma clase (0), sin diferenciar entre un prestamo que ya se pago completo (Fully Paid) y uno que sigue activo sin resultado aun (Current, Late, In Grace Period). con 878,317 prestamos Current metidos como "no default" junto a los 1,076,751 Fully Paid, el modelo terminaria aprendiendo a predecir "el prestamo no ha fallado todavia" en vez de "el prestamo se pago exitosamente". Son cosas distintas, pq un Current puede terminar en Charged Off despues. muy mal hecho nononono ese modelo se porta muy mal >:(. pero entiendo el objetivo del ejercicio pero mm nose nose mmm noseeee me da algo, no me dan ganas de seguir trbajando en eso esta mal hecho aaaaa

asi que uh... lo ignorare pq el profe no quiere q se haga ningun tipo de exclusion de filas pero bueno, ese modelo jum, hare uno como lo quiere el ejercicio y otro como no lo quiere el ejercicio

In [ ]:
# Version del profesor (literal, tal como esta en el enunciado)
dfprofe = df.copy()
dfprofe["default"] = dfprofe["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)
dfprofe["default"].value_counts(normalize=True)

# Version correcta (filtrando a resultado final conocido)
estados_validos = ["Fully Paid", "Charged Off"]
dfdeberiaser = df[df["loan_status"].isin(estados_validos)].copy()
dfdeberiaser["default"] = dfdeberiaser["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)
dfdeberiaser["default"].value_counts(normalize=True)

el `dfprofe` hace `1 if x == "Charged Off" else 0` asi tal cual, entonces mete Fully Paid, Current, Late, In Grace Period, todo eso dentro del 0. eso ya dijimos q esta mal, mezcla "pago bien" con "todavia no se sabe q va a pasar".

el `dfdeberiaser` primero filtra el df para quedarme solo con las filas donde loan_status es "Fully Paid" o "Charged Off" — bota todo lo demas (Current, Late, etc) antes de crear la columna default. ahi si aplico el mismo `1 if x == "Charged Off" else 0`, pero como ya filtre antes, el 0 solo puede ser Fully Paid xq es lo unico q queda aparte de Charged Off.

por eso es correcto: al filtrar primero, la columna default solo compara 2 resultados finales reales — pago vs default — sin meter prestamos q siguen corriendo y cuyo resultado no se sabe todavia. el apply es el mismo en las 2 versiones, la diferencia esta en el filtro `df[df["loan_status"].isin(estados_validos)]` que corre antes en dfdeberiaser, y que en dfprofe no existe.

## Selección de variables



De las columnas numericas y categoricas que quedaron despues de la limpieza de nulos, no analice las 70+ una por una a fondo por tiempo, asi que prioricé un subconjunto con tres criterios:

Primero, las variables que el enunciado menciona explicitamente como ejemplo (loan_amnt, int_rate, annual_inc, dti, fico_range_high, emp_length para numericas; purpose, home_ownership, addr_state, verification_status para categoricas), esas entran si o si porque son las que el profesor espera ver analizadas.

Segundo, variables que por conocimiento del dominio de credito se sabe que suelen ser predictoras fuertes de riesgo de impago aunque el enunciado no las mencione: installment, revol_util, open_acc, total_acc, tot_cur_bal, avg_cur_bal, delinq_2yrs, pub_rec, term y grade. Estas las meti porque son variables estandar en cualquier modelo de credit scoring (utilizacion de credito revolvente, cantidad de cuentas abiertas, historial de morosidad, calificacion de riesgo de la plataforma), no porque las haya visto destacarse antes de correr el analisis.

Tercero, exclui a proposito las que ya habia identificado en la revision de la muestra como problematicas o no aptas para este tipo de analisis: zip_code y emp_title por alta cardinalidad, las columnas de fecha porque estan en texto y no parseadas todavia, sub_grade porque es una version mas fina de grade y ya sabia que iba a salir redundante, y debt_settlement_flag porque sospechaba leakage y no queria meterla al analisis bidimensional todavia sin revisarla aparte primero.

O sea que la lista no fue exhaustiva sobre las 70+ variables numericas ni sobre todas las categoricas, fue una seleccion dirigida por lo que pide el enunciado mas conocimiento de dominio, dejando por fuera las que ya tenia razones puntuales para no incluir. Las variables numericas que no entraron fueron (bc_open_to_buy, num_actv_bc_tl, mort_acc, num_rev_accts, mo_sin_old_rev_tl_op, entre otras)

ahora, hacemos los analisis univariados y multivariados

### 9.10.4.1.2. Análisis unidimensional

Se trabaja a partir de `dfdeberiaser`, la versión correcta de la variable 
objetivo (filtrada a `Fully Paid` / `Charged Off`). Se separan las columnas 
en numéricas y categóricas, excluyendo `default` del grupo numérico y 
`loan_status`, `id`, `url` del grupo categórico por no ser variables 
predictoras.

Se calcula un resumen estadístico general (media, mediana, desviación 
estándar, mínimo, máximo, percentiles e IQR) para todas las variables 
numéricas de una sola vez, con el fin de priorizar cuáles requieren análisis 
visual individual (histograma, boxplot, evaluación de normalidad) según su 
dispersión y asimetría.

In [ ]:
# ==========================================================
# 9.10.4.1.2 - ANALISIS UNIDIMENSIONAL - SETUP
# ==========================================================
df = dfdeberiaser.copy()
df.shape

numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(include=["object"]).columns.tolist()

for col in ["default"]:
    if col in numericas:
        numericas.remove(col)
for col in ["loan_status", "id", "url"]:
    if col in categoricas:
        categoricas.remove(col)

print("Numericas:", len(numericas))
print("Categoricas:", len(categoricas))

resumen_numericas = df[numericas].describe().T
resumen_numericas["IQR"] = resumen_numericas["75%"] - resumen_numericas["25%"]
resumen_numericas

In [ ]:
print("dti == 999:", (df["dti"] == 999).sum())
print("tot_hi_cred_lim == 9999999:", (df["tot_hi_cred_lim"] == 9999999).sum())
print("revol_util > 200:", (df["revol_util"] > 200).sum())

In [ ]:
filas_antes = df.shape[0]

df = df[df["dti"] != 999]
df = df[df["tot_hi_cred_lim"] != 9999999]
df = df[df["revol_util"] <= 200]

print("Filas eliminadas:", filas_antes - df.shape[0])
df.shape

Al revisar el resumen estadístico general se detectaron valores máximos 
sospechosamente redondos en tres variables, típicos de placeholders de 
"dato desconocido" disfrazados como valor numérico real, en lugar de 
outliers genuinos:

- `dti` con valor 999 (38 filas): un ratio de deuda sobre ingreso de 999 no 
  es plausible como dato real.
- `tot_hi_cred_lim` con valor 9,999,999 (12 filas): límite de crédito 
  histórico total, mismo patrón.
- `revol_util` con valores superiores a 200% (2 filas): un ratio de 
  utilización de crédito revolving por encima de 200% es extremadamente 
  improbable y consistente con error de captura.

Dado el volumen marginal de cada caso se optó por eliminación de fila en lugar 
de imputación.

In [ ]:
# ==========================================================
# MUESTRA DE INSPECCION: 100 filas inicio / 100 medio / 100 final
# Se toma sobre el df ya limpio (post-filtrado dfdeberiaser + sentinelas)
# ==========================================================
n = df.shape[0]
mitad = n // 2

muestra_inicio = df.iloc[0:100]
muestra_medio  = df.iloc[mitad:mitad+100]
muestra_final  = df.iloc[n-100:n]

muestra_300 = pd.concat(
    [muestra_inicio, muestra_medio, muestra_final],
    keys=["inicio", "medio", "final"]
)

print("Filas totales en df:", n)
print("Columnas:", df.shape[1])
print(muestra_300.shape)

muestra_300
muestra_300.to_csv("muestra_inspeccion_300.csv")

Revise la muestra de 300 filas (100 inicio, 100 medio, 100 final) que saque del df ya limpio, para tener una idea real de como vienen los datos antes de meterme al analisis unidimensional. Encontre varias cosas que valen la pena anotar antes de seguir.

Primero, hay columnas numericas que en las 300 filas de la muestra solo tienen un valor unico: out_prncp, out_prncp_inv, policy_code y num_tl_120dpd_2m. Si esto se confirma en el dataset completo, estas columnas no sirven para nada porque no varian, entonces no pueden explicar ni un poco la diferencia entre default y no default. Las voy a marcar como candidatas a eliminar, pero antes voy a correr el mismo chequeo (nunique) sobre el dataset completo para confirmar que no es una casualidad de la muestra.

Con las categoricas paso algo parecido. pymnt_plan, hardship_flag y disbursement_method tambien salieron con un solo valor unico en toda la muestra (n, N y Cash respectivamente). Mismo caso que las numericas de varianza cero, si se confirma con el dataset completo estas tres tampoco aportan nada como variable predictora y las voy a sacar del analisis.

Otra cosa que note es que term viene con un espacio al inicio del texto (' 36 months', ' 60 months'), asi que antes de usarla para cualquier cosa toca limpiarla con un strip, si no despues me puede generar categorias duplicadas por el espacio. igual lo mejor seria solo dejar el numero para que le sirva al modelo.

Las columnas de fecha (issue_d, earliest_cr_line, last_pymnt_d, last_credit_pull_d) vienen como texto en formato Mon-YYYY, no como fecha real. Toca parsearlas con pd.to_datetime especificando el formato antes de poder sacarles algo util, como por ejemplo la antiguedad del historial crediticio a partir de earliest_cr_line.

zip_code viene truncado (tipo 190xx) y en la muestra de 300 filas ya tiene 208 valores distintos, asi que como esta no sirve de mucho para el modelo, addr_state es la que realmente puedo usar como variable geografica, pero hacer one hot encoding para esa es un poco pesado y realmente no se si aportaría gran cosa al modelo.

emp_title y title son texto libre con muchisima cardinalidad, no son las variables categoricas tipo purpose o home_ownership que pide el enunciado, asi que probablemente las voy a dejar por fuera del modelado o las trato aparte si me da tiempo.

sub_grade es basicamente una version mas fina de grade, entonces ya se que cuando llegue al analisis bidimensional y la matriz de correlacion probablemente van a salir correlacionadas fuerte entre si, eso lo voy a marcar como posible redundancia.

Por ultimo, debt_settlement_flag si tiene dos valores en la muestra (N y Y), a diferencia de las otras tres flags que solo tenian uno. Esto me preocupa un poco porque un debt settlement pasa despues de que el prestamo ya tuvo problemas, entonces sospecho que esta variable puede tener fuga de informacion (leakage) con respecto a default, osea que puede estar prediciendo el pasado en vez de ayudando a predecir el futuro. Antes de decidir si la uso o no en el modelo quiero revisar con evidencia (crosstab contra default) que tan fuerte es esa relacion.

Con esto ya tengo un mapa mas claro de por donde empezar el analisis unidimensional, primero corro los chequeos de varianza cero y valor unico sobre el dataset completo para confirmar lo que vi en la muestra, y despues sigo con histogramas y boxplots de las numericas priorizadas.

## Confirmar varianza cero / valor unico en dataset completo + drop

In [ ]:
cols_check = ['out_prncp', 'out_prncp_inv', 'policy_code', 'num_tl_120dpd_2m',
              'pymnt_plan', 'hardship_flag', 'disbursement_method']

nunique_check = df[cols_check].nunique()
print(nunique_check)

cols_sin_varianza = nunique_check[nunique_check == 1].index.tolist()
print("Columnas a eliminar por varianza cero:", cols_sin_varianza)

df = df.drop(columns=cols_sin_varianza)
df.shape

## Limpiar term a numerico

In [ ]:
df['term'] = df['term'].str.strip().str.replace(' months', '', regex=False).astype(int)
df['term'].value_counts()

## Distribucion del target

In [ ]:
target_counts = df['default'].value_counts()
target_pct = df['default'].value_counts(normalize=True) * 100
print(target_counts)
print(target_pct)

target_counts.plot(kind='bar')

## Resumen numerico priorizado + outliers IQR (loop)

Histogramas + boxplots

In [ ]:
numericas_prioridad = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
                        'fico_range_high', 'revol_util', 'open_acc', 'total_acc',
                        'tot_cur_bal', 'avg_cur_bal', 'delinq_2yrs', 'pub_rec', 'term']

resumen = df[numericas_prioridad].describe().T
resumen['IQR'] = resumen['75%'] - resumen['25%']
resumen['skew'] = df[numericas_prioridad].skew()

outliers_dict = {}
for col in numericas_prioridad:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers_dict[col] = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()

resumen['n_outliers_iqr'] = pd.Series(outliers_dict)
resumen

## Histogramas + boxplots (todas en una figura)

In [ ]:
fig, axes = plt.subplots(len(numericas_prioridad), 2, figsize=(12, 4*len(numericas_prioridad)))
for i, col in enumerate(numericas_prioridad):
    df[col].hist(bins=50, ax=axes[i,0])
    axes[i,0].set_title(f'{col} - histograma')
    df.boxplot(column=col, ax=axes[i,1])
    axes[i,1].set_title(f'{col} - boxplot')
plt.tight_layout()
plt.savefig('univariado.png', dpi=80)
plt.show()

## Categoricas: frecuencias

In [ ]:
categoricas_prioridad = ['purpose', 'home_ownership', 'verification_status', 'grade', 'addr_state']

for col in categoricas_prioridad:
    print(f"--- {col} ---")
    print(df[col].value_counts(normalize=True).round(3) * 100)
    print()

## Bidimensional rapido: numericas vs default

In [ ]:
from scipy.stats import pointbiserialr, mannwhitneyu

resultados_biv = []
for col in numericas_prioridad:
    r, p_corr = pointbiserialr(df['default'], df[col])
    stat, p_mw = mannwhitneyu(df[df['default']==0][col].dropna(), df[df['default']==1][col].dropna())
    resultados_biv.append({'variable': col, 'corr_puntobiserial': r, 'p_corr': p_corr, 'p_mannwhitney': p_mw})

pd.DataFrame(resultados_biv).sort_values('corr_puntobiserial', key=abs, ascending=False)

## Chi-cuadrado categoricas vs default

In [ ]:
from scipy.stats import chi2_contingency

for col in categoricas_prioridad:
    tabla = pd.crosstab(df[col], df['default'])
    chi2, p, dof, exp = chi2_contingency(tabla)
    print(f"{col}: chi2={chi2:.2f}, p={p:.4g}")

##  Matriz de correlacion

In [ ]:
corr = df[numericas_prioridad + ['default']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.savefig('corr_heatmap.png', dpi=80)
plt.show()
corr['default'].sort_values(key=abs, ascending=False)

Con todo el analisis unidimensional y bidimensional ya corrido, este es el resumen de lo que encontre y las decisiones que me deja para preprocesamiento.

La variable objetivo default quedo en 1,076,034 casos de clase 0 (80.04%) contra 268,367 de clase 1 (19.96%), un desbalance de aproximadamente 4 a 1. No es un desbalance extremo pero si suficiente para que el modelo tienda a favorecer la clase mayoritaria si no hago nada, asi que para el modelado voy a usar class_weight='balanced' en sklearn y el equivalente en Spark, ademas de reportar precision y recall por clase y no solo accuracy.

Sobre las variables numericas priorizadas, el resumen estadistico mas los histogramas y boxplots me mostraron distintos tipos de comportamiento. loan_amnt, int_rate, installment, fico_range_high, revol_util, open_acc y total_acc son variables continuas genuinas donde el metodo de IQR si detecta outliers reales, con distintos niveles de asimetria pero nada que requiera transformacion obligatoria para un RandomForest. annual_inc, tot_cur_bal y avg_cur_bal en cambio tienen distribuciones extremadamente concentradas cerca de cero con colas de valores hasta millones, no son errores sino datos reales en escala distinta, asi que quedan marcadas para log-transform si se llega a probar un modelo sensible a escala. delinq_2yrs, pub_rec y term son casos donde el IQR marca como outlier lo que en realidad es la clase minoritaria de una variable de conteo o casi binaria, no un error de dato, asi que esos no se tocan. Tambien encontre que dti tiene valores centinela adicionales al 999 que ya habia limpiado, llegando hasta cerca de 992, asi que ese filtro toca ampliarlo por rango antes de seguir.

En el analisis bidimensional, la correlacion punto-biserial y Mann-Whitney contra default salieron todas con p-value practicamente 0 dado el tamaño de muestra, asi que lo relevante fue la magnitud, no la significancia. int_rate fue la variable numerica mas correlacionada con default (0.259), seguida de term (0.176) y fico_range_high en negativo (-0.137, a mayor fico menor probabilidad de default, logico). dti aparecio con 0.0955 en la matriz de correlacion, mas alto de lo que esperaba dado que en el punto-biserial no se habia destacado tanto. El resto (loan_amnt, revol_util, installment, annual_inc, open_acc, pub_rec, delinq_2yrs) quedaron entre 0.02 y 0.07, señales individuales debiles pero que en conjunto pueden aportar al modelo.

De las categoricas probadas (purpose, home_ownership, verification_status, grade, addr_state), las 5 salieron con chi-cuadrado significativo (p=0), con grade muy por encima de las demas (chi2=92228.72), lo cual tiene sentido porque es la calificacion de riesgo que la plataforma ya le asigna al prestamo antes de aprobarlo.

En la matriz de correlacion general encontre tres pares con multicolinealidad fuerte: loan_amnt e installment en 0.95 (installment se calcula a partir de loan_amnt, int_rate y term, es casi la misma informacion repetida), tot_cur_bal y avg_cur_bal en 0.84, y open_acc con total_acc en 0.70. Tambien confirme que fico_range_high correlaciona negativo con int_rate (-0.41), lo que sugiere que la plataforma si usa el fico score para fijar la tasa.

Juntando todo esto, la seleccion final de variables para el modelo queda en: int_rate, term, fico_range_high, dti, grade, revol_util, annual_inc, purpose, home_ownership y verification_status como base solida, mas installment en vez de loan_amnt (para evitar el par con 0.95 de correlacion), y avg_cur_bal en vez de tot_cur_bal (para evitar el par con 0.84). open_acc y total_acc los dejo ambas por ahora ya que 0.70 no es tan extremo. Antes de pasar a preprocesamiento me falta: ampliar el filtro de valores centinela en dti (actualmente solo saque el 999 exacto, pero hay mas valores altos sospechosos)

# INICIO DE PREPROCESAMIENTO/MODELADO

tengo ese trabajo, ya hice todo lo del eda queda la parte del preprocesamiento, mira te doy una muestra del dataset para que sepas que vas a poner





TE ACLARO, EN ESTA SESION, TU NO VAZ A CORRER ABSOLUTAMENTE NADA,  SOLO ME VAS A DAR LOS IPYNB CON CELDAS Y CODIGO QUE YO VOY A COPIAR Y PEGAR EN MI IPYNB Y EJECUTAR, POR ENDE CONTEXTUALIZATE DIME QUE MAS NECESITAS SI TE FALTA ALGO ANTES DE ESCRIBIR CODIGO ETC ETC ETC



aqui te paso lo que llevo, analiza lo que hice el texto etc y seguimos con lo de preprocesamiento, pienso que mejor dejamos lo de scikit learn primero ya que no requiere mucho setup, y luego dejo lo de spark, si no que tengo que comprar unas cosas y es mas conveniente de esta forma entiendes, cool, bueno



dame las celdas que debo copiar y pegar



aqui una muestra del dataset q estoy usando pero ten en cuenta todo lo que esta escrito



In [ ]:
import gc

vars_a_borrar = ['dfprofe', 'dfdeberiaser', 'dfrevision', 'muestra_inicio',
                  'muestra_medio', 'muestra_final', 'muestra_300',
                  'resumen_nulos', 'nulos', 'nulos_pct', 'cols_eliminar',
                  'columnas_revision', 'cols_check', 'nunique_check',
                  'cols_sin_varianza', 'target_counts', 'target_pct',
                  'resumen', 'outliers_dict', 'resultados_biv', 'corr',
                  'columna', 'numericas', 'categoricas']

for v in vars_a_borrar:
    if v in globals():
        del globals()[v]

gc.collect()
print(df.shape)

## 9.10.4.2. Preprocesamiento (scikit-learn)

Variables finales seleccionadas en el EDA (celda de resumen, sección 9.10.4.1.5):

Numéricas: `int_rate`, `term`, `fico_range_high`, `dti`, `revol_util`, `annual_inc`, `installment`, `avg_cur_bal`, `open_acc`, `total_acc`.
Categórica ordinal: `grade` (A–G tiene orden real, se mapea a entero 1–7 en vez de OHE, para no perder esa jerarquía).
Categóricas nominales: `purpose`, `home_ownership`, `verification_status` (OneHotEncoder, sin orden).

Sin `addr_state` (excluida en EDA por cardinalidad). Sin `emp_length`. Sin `StandardScaler`: RandomForest particiona por umbrales de valor, es invariante a escala monótona, así que estandarizar no cambia el modelo; se justifica aquí en vez de aplicarse porque no aporta nada y agrega un paso innecesario.

In [ ]:
import pandas as pd
df = pd.read_csv('C:/Users/Enma/Desktop/Stage/Frontstage/Creations/ModelSolar/notebooks/PIDAA/Notebooks/lending_club_model_ready.csv')

In [ ]:
# ==========================================================
# 9.10.4.2.1 - PREPROCESAMIENTO (scikit-learn) - SELECCION FINAL
# ==========================================================

numericas_final = ['int_rate', 'term', 'fico_range_high', 'dti', 'revol_util',
                    'annual_inc', 'installment', 'avg_cur_bal', 'open_acc',
                    'total_acc']
categoricas_ordinal = ['grade']
categoricas_nominal = ['purpose', 'home_ownership', 'verification_status']

cols_finales = numericas_final + categoricas_ordinal + categoricas_nominal + ['default']
df_model = df[cols_finales].copy()

print(df_model.shape)
df_model.isnull().sum()

Imputación: mediana para numéricas (fit solo sobre train, aplicado a test, evita leakage). `grade` no tiene nulos (es asignado por la plataforma a todo préstamo aprobado). Las nominales tampoco deberían tener nulos en esta selección; se verifica antes de imputar.

In [ ]:
# ==========================================================
# TRAIN / TEST SPLIT (80/20 estratificado) - ANTES de imputar/encodear
# para que todo el fit del preprocesamiento sea solo sobre train
# ==========================================================
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['default'])
y = df_model['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
# ==========================================================
# IMPUTACION (mediana, fit en train, transform en train y test)
# ==========================================================
from sklearn.impute import SimpleImputer

imputer_num = SimpleImputer(strategy='median')
X_train[numericas_final] = imputer_num.fit_transform(X_train[numericas_final])
X_test[numericas_final] = imputer_num.transform(X_test[numericas_final])

X_train[categoricas_nominal] = X_train[categoricas_nominal].fillna('Missing')
X_test[categoricas_nominal] = X_test[categoricas_nominal].fillna('Missing')

X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

In [ ]:
# ==========================================================
# ENCODING
# grade -> ordinal manual (A=1 ... G=7)
# purpose, home_ownership, verification_status -> OneHotEncoder
# (sin Pipeline, tal como exige el enunciado: fit en train, transform en ambos)
# ==========================================================
from sklearn.preprocessing import OneHotEncoder

grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
X_train['grade'] = X_train['grade'].map(grade_map)
X_test['grade'] = X_test['grade'].map(grade_map)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[categoricas_nominal])

train_ohe = pd.DataFrame(
    ohe.transform(X_train[categoricas_nominal]),
    columns=ohe.get_feature_names_out(categoricas_nominal),
    index=X_train.index
)
test_ohe = pd.DataFrame(
    ohe.transform(X_test[categoricas_nominal]),
    columns=ohe.get_feature_names_out(categoricas_nominal),
    index=X_test.index
)

X_train_final = pd.concat(
    [X_train[numericas_final + categoricas_ordinal], train_ohe], axis=1
)
X_test_final = pd.concat(
    [X_test[numericas_final + categoricas_ordinal], test_ohe], axis=1
)

print(X_train_final.shape, X_test_final.shape)
X_train_final.head()

## 9.10.4.3. Modelado con scikit-learn

`RandomForestClassifier` + `GridSearchCV` sin `Pipeline`, sobre `n_estimators: [10, 50, 100]`, `max_depth: [5, 10, 15]`. `class_weight='balanced'` por el desbalance 80/20 detectado en el EDA. `cv=3` (mismo número de folds que se usará después en Spark con `CrossValidator`, para que la comparación de tiempos sea más justa). `scoring='roc_auc'` como criterio de selección del mejor modelo, dado el desbalance de clases.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, 15]
}

rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_search.fit(X_train_final, y_train)
tiempo_entrenamiento_sklearn = time.time() - t0

print(f"Tiempo entrenamiento (GridSearchCV, sklearn): {tiempo_entrenamiento_sklearn:.2f} s")
print("Mejores hiperparametros:", grid_search.best_params_)
print("Mejor ROC AUC (CV):", grid_search.best_score_)

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50],
    'max_depth': [5, 10]
}

rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=2,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_search.fit(X_train_final, y_train)
tiempo_entrenamiento_sklearn = time.time() - t0

print(f"Tiempo entrenamiento (GridSearchCV, sklearn): {tiempo_entrenamiento_sklearn:.2f} s")
print("Mejores hiperparametros:", grid_search.best_params_)
print("Mejor ROC AUC (CV):", grid_search.best_score_)

Este se realizo gracias a que pyspark no soportaba la configuración que pedia el ejercicio

In [ ]:
# ==========================================================
# EVALUACION EN TEST
# ==========================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve)

best_rf = grid_search.best_estimator_

t0 = time.time()
y_pred = best_rf.predict(X_test_final)
y_proba = best_rf.predict_proba(X_test_final)[:, 1]
tiempo_prediccion_sklearn = time.time() - t0

metricas_sklearn = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
    'tiempo_entrenamiento_s': tiempo_entrenamiento_sklearn,
    'tiempo_prediccion_s': tiempo_prediccion_sklearn
}

for k, v in metricas_sklearn.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Fully Paid', 'Charged Off']).plot(cmap='Blues')
plt.title('Matriz de confusion - RandomForest (sklearn)')
plt.savefig('confusion_matrix_sklearn.png', dpi=80)
plt.show()

In [ ]:
# ==========================================================
# EVALUACION EN TEST
# ==========================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve)

best_rf = grid_search.best_estimator_

t0 = time.time()
y_pred = best_rf.predict(X_test_final)
y_proba = best_rf.predict_proba(X_test_final)[:, 1]
tiempo_prediccion_sklearn = time.time() - t0

metricas_sklearn = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
    'tiempo_entrenamiento_s': tiempo_entrenamiento_sklearn,
    'tiempo_prediccion_s': tiempo_prediccion_sklearn
}

for k, v in metricas_sklearn.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Fully Paid', 'Charged Off']).plot(cmap='Blues')
plt.title('Matriz de confusion - RandomForest (sklearn)')
plt.savefig('confusion_matrix_sklearn.png', dpi=80)
plt.show()

In [ ]:
# ==========================================================
# CURVA ROC (se guarda para la comparacion final con Spark en 9.10.4.6)
# ==========================================================
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"RandomForest sklearn (AUC = {metricas_sklearn['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC - scikit-learn')
plt.legend()
plt.savefig('roc_sklearn.png', dpi=80)
plt.show()

In [ ]:
# ==========================================================
# GUARDAR METRICAS DE SKLEARN PARA LA COMPARACION FINAL CON SPARK (9.10.4.6)
# ==========================================================
import json

metricas_sklearn['fpr'] = fpr.tolist()
metricas_sklearn['tpr'] = tpr.tolist()

with open('metricas_sklearn.json', 'w') as f:
    json.dump(metricas_sklearn, f)

print("metricas_sklearn.json guardado con las claves:", list(metricas_sklearn.keys()))

## 9.10.4.5. Interpretabilidad con LIME (sobre el modelo de sklearn)

Se selecciona una instancia mal clasificada del set de test para explicar localmente con LIME. LIME opera en memoria local sobre arrays de numpy, por eso se aplica aquí y no directamente sobre Spark.

In [ ]:
import sys
!{sys.executable} -m pip install lime

In [ ]:

import lime
import lime.lime_tabular

errores = np.where(y_pred != y_test.values)[0]
errores = np.where(y_pred != y_test.values)[0]
proba_error = y_proba[errores]
idx_error = errores[np.argmax(np.abs(proba_error - 0.5))]
print(f"Instancia mal clasificada (alta confianza): indice {idx_error}, real={y_test.values[idx_error]}, proba_charged_off={y_proba[idx_error]:.3f}")
print(f"Instancia mal clasificada: indice {idx_error}, real={y_test.values[idx_error]}, predicho={y_pred[idx_error]}")

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_final.values,
    feature_names=X_train_final.columns.tolist(),
    class_names=['Fully Paid', 'Charged Off'],
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

exp = explainer.explain_instance(
    X_test_final.values[idx_error],
    best_rf.predict_proba,
    num_features=10
)

from IPython.display import display, HTML
display(HTML(exp.as_html(show_table=True)))

Interpretación de la instancia mal clasificada (LIME)

Instancia real = 1 (Charged Off), predicha = 0 (Fully Paid), con el modelo asignando 95% de probabilidad a "Fully Paid". Es un error de alta confianza.

El perfil de la instancia según LIME: grade = A (1), int_rate = 5.32%, fico_range_high = 814, dti = 6.53, term = 36 meses. Todas estas variables son señales de bajo riesgo crediticio y coinciden con lo que el modelo aprendió como perfil "seguro" durante el entrenamiento.

El préstamo terminó en default de todas formas, lo cual evidencia una limitación estructural del modelo: las variables disponibles (grade, int_rate, fico, dti, etc.) describen el riesgo al momento de la aprobación, pero no capturan eventos posteriores que ocurren durante la vida del préstamo (pérdida de empleo, gasto médico imprevisto, divorcio, u otro shock financiero). LIME explica la predicción en función de lo que el modelo efectivamente observó, no puede señalar la información ausente que habría cambiado el resultado. Esto es consistente con lo esperado: grade e int_rate son en sí mismos un estimado de riesgo hecho por Lending Club en el momento de originación, y aun ese estimado experto falla en instancias individuales.

# Inicio SPARK

In [ ]:
# ==========================================================
# DIAGNOSTICO SPARK
# ==========================================================
import sys, os, subprocess, platform

print("=== 1. SISTEMA ===")
print("OS:", platform.platform())
print("Python:", sys.version)
print("Python exe:", sys.executable)

print("\n=== 2. RAM / CPU ===")
try:
    import psutil
    print(f"RAM total: {psutil.virtual_memory().total / (1024**3):.1f} GB")
    print(f"RAM disponible ahora: {psutil.virtual_memory().available / (1024**3):.1f} GB")
    print(f"Cores logicos: {psutil.cpu_count(logical=True)}")
    print(f"Cores fisicos: {psutil.cpu_count(logical=False)}")
except ImportError:
    print("psutil no instalado -> !{sys.executable} -m pip install psutil")

print("\n=== 3. JAVA ===")
try:
    out = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print(out.stderr or out.stdout)
except FileNotFoundError:
    print("java NO esta en el PATH")
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NO DEFINIDA"))

print("\n=== 4. PYSPARK ===")
try:
    import pyspark
    print("pyspark version:", pyspark.__version__)
    print("pyspark path:", pyspark.__file__)
except ImportError:
    print("pyspark NO instalado -> !{sys.executable} -m pip install pyspark")

print("\n=== 5. HADOOP / WINUTILS (necesario en Windows) ===")
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME", "NO DEFINIDA"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME", "NO DEFINIDA"))
hadoop_home = os.environ.get("HADOOP_HOME")
if hadoop_home:
    winutils_path = os.path.join(hadoop_home, "bin", "winutils.exe")
    print("winutils.exe existe:", os.path.exists(winutils_path), "->", winutils_path)
else:
    print("winutils.exe: no se puede chequear, HADOOP_HOME no definida")

print("\n=== 6. PRUEBA REAL: crear SparkSession y correr una accion minima ===")
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("DiagnosticoTest") \
        .master("local[*]") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    resultado = spark.range(5).count()
    print("SparkSession OK, spark.range(5).count() =", resultado)
    print("Spark version:", spark.version)
    spark.stop()
except Exception as e:
    import traceback
    print("FALLO al crear/usar SparkSession:")
    traceback.print_exc()

In [ ]:
import os, glob

pyspark_jars = os.path.join(os.path.dirname(__import__("pyspark").__file__), "jars")
hadoop_jars = [f for f in os.listdir(pyspark_jars) if "hadoop-client-api" in f or "hadoop-common" in f]
print(hadoop_jars)

In [ ]:
import subprocess
out = subprocess.run(["where", "java"], capture_output=True, text=True)
print(out.stdout)

In [ ]:
df_model.to_csv('lending_club_model_ready.csv', index=False)
print(df_model.shape)

## 9.10.4.2.2. Preprocesamiento (PySpark)

**Justificación de la configuración de sesión (desviación del enunciado):**

El enunciado pide `spark.executor.memory=8g` + `spark.driver.memory=8g` (16 GB solo para Spark). La máquina de desarrollo tiene 13.8 GB de RAM totales, con 4-7 GB libres en promedio durante las pruebas. Esa config es inviable aquí y produciría fallas de asignación de memoria o swapping severo.

Además, en `local[*]` no existe un proceso *executor* separado del *driver* — todo corre en la misma JVM, así que `spark.executor.memory` no tiene efecto real en este modo (es una config pensada para cluster). Por eso se documenta pero no se usa como límite operativo.

`spark.sql.shuffle.partitions` y `spark.default.parallelism` se bajan de 400 a un valor proporcional a los cores disponibles (8 físicos / 16 lógicos). 400 particiones en una máquina de 8 cores generaría miles de tareas mínimas con overhead de scheduling mayor al trabajo real por partición, más lento que sin ese ajuste.

Config usada: `driver.memory=4g` (único valor real que importa en local), `shuffle.partitions=32`, `default.parallelism=16`, resto de fracciones de memoria igual al enunciado (son proporciones del heap, no cantidades absolutas, no generan riesgo).

**Nota sobre la configuracion de Spark (ajuste durante la ejecucion):**

En una primera corrida, con la sesion de Spark configurada como arriba (`shuffle.partitions=32`, `driver.memory=4g`) pero con `numFolds=3` y `parallelism=4` en el `CrossValidator` (4 modelos de RandomForest entrenandose de forma simultanea), la JVM de Spark se quedo sin memoria y el proceso murio a los ~8 minutos de entrenamiento (`Py4JNetworkError` / `ConnectionRefusedError` al perderse la conexion con la JVM). Entrenar 4 RandomForest en paralelo sobre ~1.3 millones de filas, cada uno generando su propio arbol de tareas/particiones, satura la memoria disponible en una maquina local con recursos limitados (8 cores fisicos, 13.8 GB de RAM total).

Se ajusto el `CrossValidator` a `parallelism=1` (un modelo a la vez, en vez de 4 simultaneos) y `numFolds=2` (permitido por el enunciado, en vez de 3). Esto evito que volviera a caerse, a costa de un mayor tiempo de pared por no paralelizar el ajuste de hiperparametros. Este es en si mismo un hallazgo relevante para la comparacion de la seccion 9.10.4.6: scikit-learn con `n_jobs=-1` uso los cores locales de forma directa y eficiente sin fallar, mientras que Spark, disenado para clusters distribuidos, requirio reducir la paralelizacion del ajuste de hiperparametros para correr de forma estable en una sola maquina con memoria limitada.

**Segundo ajuste (grilla reducida):** incluso con `parallelism=1` y `numFolds=2`, la JVM volvio a caerse con `ConnectionRefusedError` al intentar entrenar la grilla completa (`numTrees: [10,50,100]`, `maxDepth: [5,10,15]` = 9 combinaciones x 2 folds = 18 modelos). Cada arbol de `maxDepth=15` sobre ~1.3 millones de filas, entrenado en una sola JVM local compartiendo memoria con el driver y el resto del pipeline, es suficiente para agotar los 4 GB asignados sin necesidad de paralelismo entre modelos.

Se redujo la grilla a `numTrees: [10, 50]` y `maxDepth: [5, 10]` (4 combinaciones x 2 folds = 8 modelos), manteniendo obligatoriamente `ParamGridBuilder` + `CrossValidator` (sin busqueda manual por bucles, tal como exige el enunciado). Esto NO es una grilla identica en tamano a la usada en scikit-learn (`n_estimators: [10,50,100]`, `max_depth: [5,10,15]`), y se documenta como limitacion practica de hardware, no como eleccion metodologica: el enunciado exige el mecanismo de busqueda (`ParamGridBuilder`+`CrossValidator`), no un tamano de grilla identico entre ambos frameworks. El mejor modelo encontrado por `CrossValidator` (`cv_model.bestModel`) y su `ROC AUC` promedio (`cv_model.avgMetrics`) se reportan igual en la seccion de metricas y en la comparacion final (9.10.4.6).

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

RANDOM_STATE = 99  # debe coincidir con el RANDOM_STATE usado en la parte de sklearn

spark = SparkSession.builder \
    .appName("LendingClub_Optimized") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.default.parallelism", "16") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.fraction", 0.8) \
    .config("spark.memory.storageFraction", 0.3) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Cores disponibles (local):", spark.sparkContext.defaultParallelism)

In [ ]:
# ==========================================================
# CARGA DEL DATASET YA CURADO EN EL EDA (mismas filas/columnas que uso sklearn)
# Prohibido .toPandas()/.collect() sobre el dataset completo: NO se hace aqui.
# ==========================================================
t0 = time.time()
sdf = spark.read.csv('lending_club_model_ready.csv', header=True, inferSchema=True)
tiempo_carga_spark = time.time() - t0

print(f"Tiempo de carga: {tiempo_carga_spark:.2f} s")
print("Filas:", sdf.count(), "| Columnas:", len(sdf.columns))
sdf.printSchema()

**Encoding (version corregida, sin data leakage):**
- `grade` (A-G, ordinal real) -> mapeo manual fijo a entero 1-7 (no se "aprende" de los datos, es un diccionario fijo A=1...G=7, por eso se puede aplicar antes del split sin ningun riesgo de fuga).
- `purpose`, `home_ownership`, `verification_status` (nominales) -> `StringIndexer` + `OneHotEncoder`, **ajustados (`fit`) unicamente sobre el set de train, y aplicados (`transform`) tanto a train como a test**. Es el mismo criterio que se uso en sklearn (`OneHotEncoder.fit(X_train)`, `transform` en ambos), para que ninguna categoria del set de test influya en como se codifican las variables durante el entrenamiento.
- Por eso el split 80/20 se hace **antes** del `StringIndexer`/`OneHotEncoder` (mas abajo), y no despues como en una primera version de este notebook.
- `handleInvalid="keep"` en el `StringIndexer` cubre el caso borde de que aparezca en test una categoria que no estaba en train (le asigna un indice adicional en vez de fallar).
- Sin `StandardScaler`: mismo argumento que en sklearn, RandomForest es invariante a la escala de las features.
- `class_weight='balanced'` no existe en `pyspark.ml.classification.RandomForestClassifier`. Se reemplaza con una columna `classWeight` calculada **solo con la distribucion de clases del set de train** y pasada a `weightCol`, que es el mecanismo equivalente en Spark ML.

In [ ]:
# ==========================================================
# GRADE -> ORDINAL MANUAL (igual mapeo que en sklearn)
# ==========================================================
sdf = sdf.withColumn(
    "grade_num",
    F.when(F.col("grade") == "A", 1)
     .when(F.col("grade") == "B", 2)
     .when(F.col("grade") == "C", 3)
     .when(F.col("grade") == "D", 4)
     .when(F.col("grade") == "E", 5)
     .when(F.col("grade") == "F", 6)
     .when(F.col("grade") == "G", 7)
     .otherwise(None)
     .cast(T.DoubleType())
)

In [ ]:
# ==========================================================
# SPLIT 80/20 ESTRATIFICADO POR CLASE -- ANTES de ajustar cualquier
# encoder (StringIndexer/OneHotEncoder), para que el set de test no
# influya en el preprocesamiento. Mismo criterio que en sklearn:
# fit solo en train, transform en train y test.
# ==========================================================
sdf_0 = sdf.filter(F.col("default") == 0)
sdf_1 = sdf.filter(F.col("default") == 1)

train_0, test_0 = sdf_0.randomSplit([0.8, 0.2], seed=RANDOM_STATE)
train_1, test_1 = sdf_1.randomSplit([0.8, 0.2], seed=RANDOM_STATE)

train_raw = train_0.union(train_1)
test_raw = test_0.union(test_1)

print("Train (raw):", train_raw.count(), "| Test (raw):", test_raw.count())
train_raw.groupBy("default").count().show()
test_raw.groupBy("default").count().show()

In [ ]:
# ==========================================================
# NOMINALES -> StringIndexer + OneHotEncoder
# Se ajustan (fit) SOLO sobre train_raw y se aplican (transform) a
# train y test -- igual criterio que el OneHotEncoder de sklearn
# (fit en train, transform en ambos). handleInvalid="keep" cubre
# categorias que aparezcan en test y no existan en train.
# ==========================================================
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

categoricas_nominal = ['purpose', 'home_ownership', 'verification_status']

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categoricas_nominal
]
encoders = [
    OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe")
    for c in categoricas_nominal
]

encoding_pipeline = Pipeline(stages=indexers + encoders)
encoding_model = encoding_pipeline.fit(train_raw)  # fit SOLO en train

train_idx = encoding_model.transform(train_raw)
test_idx = encoding_model.transform(test_raw)

train_idx.select([c + "_ohe" for c in categoricas_nominal]).show(5, truncate=False)

In [ ]:
# ==========================================================
# COLUMNA DE PESO POR CLASE (equivalente a class_weight='balanced' de sklearn)
# calculada SOLO con la distribucion de clases de TRAIN (no de test)
# ==========================================================
conteo_clases = train_idx.groupBy("default").count().collect()
total = sum(r["count"] for r in conteo_clases)
n_clases = len(conteo_clases)
pesos = {r["default"]: total / (n_clases * r["count"]) for r in conteo_clases}
print("Pesos por clase (calculados solo en train):", pesos)

map_expr = F.create_map([F.lit(x) for kv in pesos.items() for x in kv])
train_idx = train_idx.withColumn("classWeight", map_expr[F.col("default")])

In [ ]:
# ==========================================================
# VECTOR ASSEMBLER + CACHE (obligatorio antes del modelo)
# Mismo assembler aplicado a train y test (VectorAssembler no
# "aprende" nada de los datos, solo ensambla columnas en un vector,
# por eso no hay riesgo de fuga en este paso).
# ==========================================================
from pyspark.ml.feature import VectorAssembler

numericas_final = ['int_rate', 'term', 'fico_range_high', 'dti', 'revol_util',
                    'annual_inc', 'installment', 'avg_cur_bal', 'open_acc',
                    'total_acc']

feature_cols = numericas_final + ['grade_num'] + [c + "_ohe" for c in categoricas_nominal]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

train_df = assembler.transform(train_idx).select(
    "features", "classWeight", F.col("default").cast("double").alias("label")
)
test_df = assembler.transform(test_idx).select(
    "features", F.col("default").cast("double").alias("label")
)

train_df.cache()
test_df.cache()

print("Train:", train_df.count(), "| Test:", test_df.count())
train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()

## 9.10.4.4. Modelado con PySpark

`RandomForestClassifier` + `ParamGridBuilder` + `CrossValidator(numFolds=2)`, sin bucles manuales sobre hiperparámetros, tal como exige el enunciado. Grilla reducida respecto a sklearn (`numTrees: [10, 50]`, `maxDepth: [5, 10]`) por restricción de hardware local — ver nota de configuración más arriba con el detalle de los dos crashes por memoria que llevaron a este ajuste. `weightCol='classWeight'` para el balance de clases. `metricName='areaUnderROC'` en el evaluador, igual criterio que `scoring='roc_auc'` usado en `GridSearchCV`.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

rf_spark = RandomForestClassifier(
    featuresCol="features", labelCol="label", weightCol="classWeight", seed=RANDOM_STATE
)

param_grid_spark = (
    ParamGridBuilder()
    .addGrid(rf_spark.numTrees, [10, 50])
    .addGrid(rf_spark.maxDepth, [5, 10])
    .build()
)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=rf_spark,
    estimatorParamMaps=param_grid_spark,
    evaluator=evaluator_auc,
    numFolds=2,
    seed=RANDOM_STATE,
    parallelism=1
)

t0 = time.time()
cv_model = cv.fit(train_df)
tiempo_entrenamiento_spark = time.time() - t0

best_model = cv_model.bestModel
print(f"Tiempo entrenamiento (CrossValidator, Spark): {tiempo_entrenamiento_spark:.2f} s")
print("Mejor numTrees:", best_model.getNumTrees)
print("Mejor maxDepth:", best_model._java_obj.getMaxDepth())
print("Mejor ROC AUC (CV):", max(cv_model.avgMetrics))

In [ ]:
# ==========================================================
# EVALUACION EN TEST
# ==========================================================
t0 = time.time()
predictions = cv_model.transform(test_df)
predictions.cache()
n_pred = predictions.count()  # fuerza la evaluacion (Spark es lazy)
tiempo_prediccion_spark = time.time() - t0

roc_auc_spark = evaluator_auc.evaluate(predictions)
print(f"Tiempo prediccion: {tiempo_prediccion_spark:.2f} s")
print(f"ROC AUC (test): {roc_auc_spark:.4f}")

In [ ]:
# ==========================================================
# ACCURACY / PRECISION / RECALL / F1 (clase positiva = 1, Charged Off)
# ==========================================================
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_multi = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy_spark = evaluator_multi.setMetricName("accuracy").evaluate(predictions)
precision_spark = evaluator_multi.setMetricName("precisionByLabel").setMetricLabel(1.0).evaluate(predictions)
recall_spark = evaluator_multi.setMetricName("recallByLabel").setMetricLabel(1.0).evaluate(predictions)
f1_spark = evaluator_multi.setMetricName("fMeasureByLabel").setMetricLabel(1.0).evaluate(predictions)

metricas_spark = {
    'accuracy': accuracy_spark,
    'precision': precision_spark,
    'recall': recall_spark,
    'f1': f1_spark,
    'roc_auc': roc_auc_spark,
    'tiempo_entrenamiento_s': tiempo_entrenamiento_spark,
    'tiempo_prediccion_s': tiempo_prediccion_spark,
    'best_numTrees': int(best_model.getNumTrees),
    'best_maxDepth': int(best_model._java_obj.getMaxDepth())
}

for k, v in metricas_spark.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

evaluator_binary = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction"
)

roc_auc_spark = evaluator_binary.evaluate(predictions, {evaluator_binary.metricName: "areaUnderROC"})
pr_auc_spark = evaluator_binary.evaluate(predictions, {evaluator_binary.metricName: "areaUnderPR"})

evaluator_multi = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction"
)

accuracy_spark = evaluator_multi.setMetricName("accuracy").evaluate(predictions)
precision_spark = evaluator_multi.setMetricName("precisionByLabel").setMetricLabel(1.0).evaluate(predictions)
recall_spark = evaluator_multi.setMetricName("recallByLabel").setMetricLabel(1.0).evaluate(predictions)
f1_spark = evaluator_multi.setMetricName("fMeasureByLabel").setMetricLabel(1.0).evaluate(predictions)

metricas_spark = {
    'accuracy': accuracy_spark,
    'precision': precision_spark,
    'recall': recall_spark,
    'f1': f1_spark,
    'roc_auc': roc_auc_spark,
    'pr_auc': pr_auc_spark,
    'tiempo_entrenamiento_s': tiempo_entrenamiento_spark,
    'tiempo_prediccion_s': tiempo_prediccion_spark,
    'best_numTrees': int(best_model.getOrDefault(best_model.numTrees)),
    'best_maxDepth': int(best_model.getOrDefault(best_model.maxDepth))
}

for k, v in metricas_spark.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

Sin diferencias

In [ ]:
# ==========================================================
# MATRIZ DE CONFUSION MANUAL
# ==========================================================
import matplotlib.pyplot as plt
cm_spark = predictions.groupBy("label", "prediction").count().toPandas()
cm_pivot = cm_spark.pivot(index="label", columns="prediction", values="count").fillna(0)
print(cm_pivot)

plt.figure(figsize=(5, 4))
plt.imshow(cm_pivot.values, cmap='Blues')
plt.xticks([0, 1], ['Pred: Fully Paid', 'Pred: Charged Off'])
plt.yticks([0, 1], ['Real: Fully Paid', 'Real: Charged Off'])
for i in range(cm_pivot.shape[0]):
    for j in range(cm_pivot.shape[1]):
        plt.text(j, i, int(cm_pivot.values[i, j]), ha='center', va='center')
plt.title('Matriz de confusion - RandomForest (Spark)')
plt.colorbar()
plt.savefig('confusion_matrix_spark.png', dpi=80)
plt.show()

In [ ]:
# ==========================================================
# CURVA ROC (Spark) - toPandas() del subconjunto score/label del set de
# test YA PREDICHO (no es el dataset completo ni es antes del entrenamiento,
# es el mismo tipo de operacion ya usada en la matriz de confusion y en LIME)
# ==========================================================
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc
import pyspark.sql.functions as F

pred_pd = predictions.select(
    vector_to_array("probability")[1].alias("score"),
    F.col("label")
).toPandas()

fpr_spark, tpr_spark, _ = roc_curve(pred_pd["label"], pred_pd["score"])
roc_auc_spark = auc(fpr_spark, tpr_spark)

print("Puntos en la curva ROC:", len(fpr_spark))
print(f"AUC (Spark): {roc_auc_spark:.4f}")

plt.figure(figsize=(7, 6))
plt.plot(fpr_spark, tpr_spark, label=f"RandomForest Spark (AUC = {roc_auc_spark:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC - PySpark')
plt.legend()
plt.savefig('roc_spark.png', dpi=80)
plt.show()

## 9.10.4.5. Interpretabilidad con LIME sobre el modelo de Spark

**Nota de cumplimiento:** las siguientes celdas hacen `.toPandas()` sobre subconjuntos pequeños (una muestra de fondo para el explainer, una instancia puntual), nunca sobre el dataset completo ni antes del entrenamiento. El entrenamiento ya terminó en la celda anterior; esto es solo para la explicación local de una predicción.

LIME necesita evaluar la función de predicción sobre miles de perturbaciones sintéticas de una instancia. `predict_fn` aquí tiene que: tomar el array de numpy que genera LIME, convertirlo a `pandas`, subirlo a un `DataFrame` de Spark, correr `transform()` a través del modelo distribuido, y bajar el resultado de vuelta a `pandas`/`numpy`. Cada llamada implica ese viaje redondo entre el proceso Python y la JVM de Spark, algo que en scikit-learn es una simple operación vectorizada en memoria (`predict_proba`). Se mide el tiempo exacto para cuantificarlo.

In [ ]:
# ==========================================================
# FEATURE NAMES desde los metadatos del vector ensamblado
# ==========================================================
metadata = train_df.schema["features"].metadata
attrs = metadata.get("ml_attr", {}).get("attrs", {})

all_attrs = []
for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in attrs:
        all_attrs.extend(attrs[attr_type])

all_attrs = sorted(all_attrs, key=lambda x: x["idx"])
feature_names_spark = [attr["name"] for attr in all_attrs]

print(len(feature_names_spark), "features")
print(feature_names_spark)

In [ ]:
# ==========================================================
# MUESTRA DE FONDO PARA LIME (subset pequeno, NO el dataset completo)
# ==========================================================
test_sample_pdf = predictions.filter(predictions["label"] != predictions["prediction"]).limit(500).toPandas()

if test_sample_pdf.empty:
    raise ValueError("No se encontraron errores de clasificacion en el limite evaluado. Incremente el tamano de la muestra o filtre directamente sobre el DataFrame.")

X_test_sample = np.array([row.toArray() for row in test_sample_pdf["features"]])
y_test_sample = test_sample_pdf["label"].values
y_pred_sample = test_sample_pdf["prediction"].values

idx_error_spark = 0
print(f"Instancia mal clasificada: indice {idx_error_spark}, real={y_test_sample[idx_error_spark]}, predicho={y_pred_sample[idx_error_spark]}")

In [ ]:
# ==========================================================
# predict_fn que envuelve el modelo de Spark (round-trip Python <-> JVM)
# ==========================================================
from pyspark.ml.linalg import Vectors, VectorUDT

def spark_predict_proba(X_batch):
    rows = [(Vectors.dense(x),) for x in X_batch]
    batch_sdf = spark.createDataFrame(rows, ["features"])
    result = best_model.transform(batch_sdf).select("probability").toPandas()
    return np.array([p.toArray() for p in result["probability"]])

t0 = time.time()
_ = spark_predict_proba(X_test_sample[:5])
tiempo_predict_fn_spark = time.time() - t0
print(f"Tiempo de 1 llamada a predict_fn (5 filas, via Spark): {tiempo_predict_fn_spark:.3f} s")
print(f"Comparar con sklearn: best_rf.predict_proba en el mismo numero de filas es practicamente instantaneo (<0.01 s tipico).")

In [ ]:
# ==========================================================
# LIME sobre el modelo de Spark
# ==========================================================
import lime
import lime.lime_tabular

explainer_spark = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_sample,
    feature_names=feature_names_spark,
    class_names=['Fully Paid', 'Charged Off'],
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

t0 = time.time()
exp_spark = explainer_spark.explain_instance(
    X_test_sample[idx_error_spark],
    spark_predict_proba,
    num_features=10,
    num_samples=1000  # reducido de los 5000 default de LIME para mantener el tiempo manejable
)
tiempo_lime_spark = time.time() - t0

print(f"Tiempo total explain_instance (Spark, num_samples=1000): {tiempo_lime_spark:.2f} s")

from IPython.display import display, HTML
display(HTML(exp_spark.as_html(show_table=True)))

### Limitaciones de LIME sobre modelos de Spark (discusión)

`LimeTabularExplainer` fue diseñado para funciones de predicción en memoria local (arrays de numpy in-process). Para explicar una instancia genera miles de perturbaciones sintéticas y evalúa la función de predicción sobre todas ellas. Con `sklearn` esto es una llamada vectorizada de `predict_proba` sobre un array de numpy, típicamente sub-segundo. Con Spark, cada llamada a `predict_fn` implica serializar el batch, subirlo como `DataFrame` distribuido, ejecutar el pipeline a través de la JVM, y bajar el resultado de vuelta. El tiempo medido en `tiempo_predict_fn_spark` y `tiempo_lime_spark` arriba lo cuantifica directamente en esta máquina.

Esto obliga a reducir `num_samples` muy por debajo del default de LIME (se usó 1000 en vez de 5000) para que la explicación termine en un tiempo razonable, lo que a su vez reduce la calidad/estabilidad estadística de la aproximación local que hace LIME (menos perturbaciones, superficie de decisión local peor muestreada). Es un trade-off directo: fidelidad de la explicación vs. tiempo de cómputo, que en `sklearn` prácticamente no existe porque el modelo completo vive en el mismo proceso que LIME.

La alternativa práctica para explicabilidad a escala de Spark es no depender de LIME en absoluto y usar la importancia de features nativa del modelo (`best_model.featureImportances`), que es global en vez de local, pero no requiere ningún round-trip por instancia.

In [ ]:
# feature importances nativas de Spark, como alternativa global a LIME
importancias = pd.DataFrame({
    'feature': feature_names_spark,
    'importancia': best_model.featureImportances.toArray()
}).sort_values('importancia', ascending=False)

importancias.head(10)

## 9.10.4.6. Comparación de resultados: scikit-learn vs PySpark

In [ ]:
# ==========================================================
# TABLA COMPARATIVA
# ==========================================================
import json

with open('metricas_sklearn.json') as f:
    resultados_sklearn = json.load(f)

tabla_comparativa = pd.DataFrame({
    'Metrica': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC AUC',
                'Tiempo entrenamiento (s)', 'Tiempo prediccion (s)'], 
    'scikit-learn': [
        resultados_sklearn['accuracy'], resultados_sklearn['precision'],
        resultados_sklearn['recall'], resultados_sklearn['f1'],
        resultados_sklearn['roc_auc'], resultados_sklearn['tiempo_entrenamiento_s'],
        resultados_sklearn['tiempo_prediccion_s']
    ],
    'PySpark': [
        metricas_spark['accuracy'], metricas_spark['precision'],
        metricas_spark['recall'], metricas_spark['f1'],
        metricas_spark['roc_auc'], metricas_spark['tiempo_entrenamiento_s'],
        metricas_spark['tiempo_prediccion_s']
    ]
})

tabla_comparativa

En general pyspark tiene un rendimiento reducido a comparación de scikit learn. hay que tener en cuena que no se pudo correr el tercer ajuste de kfolds por limitaciones del JVM.

In [ ]:
# ==========================================================
# GRAFICO COMPARATIVO DE TIEMPOS
# ==========================================================
labels_tiempo = ['Entrenamiento', 'Prediccion']
tiempos_sklearn = [resultados_sklearn['tiempo_entrenamiento_s'], resultados_sklearn['tiempo_prediccion_s']]
tiempos_spark = [metricas_spark['tiempo_entrenamiento_s'], metricas_spark['tiempo_prediccion_s']]

x = np.arange(len(labels_tiempo))
width = 0.35

plt.figure(figsize=(7, 5))
plt.bar(x - width/2, tiempos_sklearn, width, label='scikit-learn')
plt.bar(x + width/2, tiempos_spark, width, label='PySpark')
plt.xticks(x, labels_tiempo)
plt.ylabel('Segundos')
plt.title('Comparacion de tiempos de computo')
plt.legend()
plt.savefig('comparacion_tiempos.png', dpi=80)
plt.show()

Otra vez, incomparable, en los intentos de que pyspark trabajara con las mísmas configuraciones de scikit learn, tomaba más de 12 minutos, y eventualmente el JVM se crasheaba y era necesario reiniciar el kernel de python. Esto ocurrio 6 veces antes de encontrar las soluciones.

In [ ]:
# ==========================================================
# ROC OVERLAY FINAL
# ==========================================================
plt.figure(figsize=(7, 6))
plt.plot(resultados_sklearn['fpr'], resultados_sklearn['tpr'],
         label=f"scikit-learn (AUC = {resultados_sklearn['roc_auc']:.3f})")
plt.plot(fpr_spark, tpr_spark, label=f"PySpark (AUC = {roc_auc_spark:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC - scikit-learn vs PySpark')
plt.legend()
plt.savefig('roc_comparativa.png', dpi=80)
plt.show()

spark.stop()

Realmente el rendimiento fue casi identico, asi que podemos indagar que gracias al objetivo de uso de pyspark es una herramienta excelente, ya que es usado para cuando los datasets no pueden entrar en la memoria RAM. es decir, trabajo de datasets de mas de 200 GB en general. Asumiendo un entorno profesional.

# Conclusiones

¿Qué entorno fue más rápido y por qué? (Considerar el efecto del tamaño completo del dataset, el caché, el CrossValidator y la configuración de Spark)



Independientemente del tamaño del dataset,  Scikit learn, adaptado para cumplir exactamente las mimsas condiciones que pyspark, tuvo un rendimiento de

accuracy: 0.6313
precision: 0.3087
recall: 0.6833
f1: 0.4253
roc_auc: 0.7090
tiempo_entrenamiento_s: 112.0061
tiempo_prediccion_s: 3.9835

Y en pyspark

accuracy: 0.6270
precision: 0.3098
recall: 0.6899
f1: 0.4276
roc_auc: 0.7089
tiempo_entrenamiento_s: 84.0184
tiempo_prediccion_s: 2.6553
best_numTrees: 50
best_maxDepth: 10

Por lo que podemos ver que pyspark fue mas rapido en la ejecución.

SIN EMBARGO, pyspark no fue capaz de ejecutar las configuraciones requeridas por la actividad, además, no se aplicaron las configuraciones en su totalidad debido a que el dispositivo usado para correr el codigo no tenia suficiente memoria para que el JVM de pyspark funcionario correctamente.

¿Cuál fue más preciso?


Realmente las diferencias son minimas, menos del 1%


¿A partir de qué volumen de datos PySpark comienza a superar a scikit-learn?

Dado el numero de datos, y la forma de funcionamiento de pyspark, y el objetivo de la creacion de pyspark (Big data) pyspark empieza a funcionar mejor en datos que no quepan en memoria, sin embargo en datos pequeños, haciendo que funcione con los datos en memoria, es mas rapido que scikit, seguramente gracias al paralelismo que permite el entorno.


¿Qué aporta LIME y cuáles son sus limitaciones en entornos distribuidos?


Una explicación aproximada de que variables y en que medida afecta la decision del modelo, pero solo con una predicción en particular, es decir que hace falta escoger la predicción a evaluar. Totalmente ineficiente en entornos distribuidos, en la consulta de como usar LIME con pyspark (por que LIME esta practicamente diseñado para ser usado con scikit) nos dimos cuenta de que en realidad es un proyecto bastante particular al cual aparentemente es necesario esa compatibilidad con entornos como pyspark.


¿Qué efecto tuvo cada una de las condiciones obligatorias en el rendimiento?

Era imposible correr las condiciones obligatorias de Pyspark en esta maquina, asi que se adaptaron, un proceso que tomo varios dias. Las condiciones obligatorias en scikit no tuvieron problema, en realidad fue hasta mas conveniente su uso que pyspark, pero pyspark no esta pensado para ser usado de esta forma, en realidad es injusto hacer que pyspark, diseñado para ser usado en clusteres, se use en una sola maquina y tiene sentido que tuviese problemas. Hay que recalcar que de todas formas el paralelismo de pyspark le dio una ventaja de velocidad de casi del 30% frente a scikit, asi que por lo menos eso funciono.

# 